# The 2020 derecho, field by field: a difference-in-differences event study

**Question.** How much corn canopy did the 2020-08-10 derecho strip, per field,
and does damage scale with USDA's measured wind bands?

**Why not raw before/after?** Iowa was drying out across the exact study window
(US Drought Monitor D1 coverage rose 34.3% to 60.9%), so a raw NDVI drop
attributes drought to wind. The design: treated = corn fields inside a USDA
wind-gust polygon; controls = corn fields outside every polygon within a
matched latitude band (0.15 deg) of each treated field; the estimate is
delta(treated) minus delta(matched controls) per wind class.

All thresholds were pre-registered in `config.yml` before results were computed.
Every number derives from the Iceberg tables; nothing is loaded from files.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from config import EVENT, ICEBERG, QUALITY
from session import get_sedona
CAT = ICEBERG['catalog']
sedona = get_sedona('derecho_event_study')
sedona.sparkContext.setLogLevel('ERROR')

:: loading settings :: url = jar:file:/Users/ross/s2-field-ndvi/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/ross/.ivy2/cache
The jars for the packages stored in: /Users/ross/.ivy2/jars
org.apache.sedona#sedona-spark-shaded-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8b621c43-cc36-4098-aa04-580d807b30d7;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-shaded-3.5_2.12;1.9.1 in central
	found org.datasyslab#geotools-wrapper;1.9.1-33.5 in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.11.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 95ms :: artifacts dl 3ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from centra

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
frame = sedona.sql(f"""
    WITH wide AS (
      SELECT field_id,
             MAX(CASE WHEN date = DATE'{EVENT['pre_date']}'  THEN mean_ndvi END) AS pre,
             MIN(CASE WHEN date = DATE'{EVENT['pre_date']}'  THEN valid_frac END) AS pre_vf,
             MAX(CASE WHEN date = DATE'{EVENT['post_date']}' THEN mean_ndvi END) AS post,
             MIN(CASE WHEN date = DATE'{EVENT['post_date']}' THEN valid_frac END) AS post_vf
      FROM {CAT}.crop.field_ndvi GROUP BY field_id
    )
    SELECT w.field_id, w.post - w.pre AS delta,
           w.pre, w.post, w.pre_vf, w.post_vf,
           ST_Y(ST_Centroid(ST_GeomFromWKB(f.geom_4326_wkb))) AS lat,
           COALESCE(MAX(z.gust_class), 0) AS wind
    FROM wide w
    JOIN {CAT}.crop.fields f USING (field_id)
    LEFT JOIN {CAT}.crop.wind_zones z
      ON ST_Intersects(ST_Centroid(ST_GeomFromWKB(f.geom_4326_wkb)),
                       ST_GeomFromWKB(z.wkb_4326))
    WHERE f.CDL2020 = 1
    GROUP BY w.field_id, w.pre, w.post, w.pre_vf, w.post_vf, f.geom_4326_wkb
""").toPandas()
vf_min = float(QUALITY['valid_frac_min'])
frame['usable'] = (frame.pre.notna() & frame.post.notna()
                   & (frame.pre_vf >= vf_min) & (frame.post_vf >= vf_min))
frame[frame.usable].groupby('wind').agg(n=('delta','size'), mean_delta=('delta','mean')).round(4)

,n,mean_delta
wind,,
0,74,0.0028
1,527,-0.0116
2,436,-0.0781
3,954,-0.1043


## Difference-in-differences

Each treated field is compared against the mean delta of control fields
(wind class 0) within its latitude band; the DiD is the treated field's delta
minus its matched-control mean, averaged per wind class.

In [3]:
import pandas as pd
band = float(EVENT['control_lat_band_deg'])
controls = frame[frame.usable & (frame.wind == 0)][['lat', 'delta']].sort_values('lat').reset_index(drop=True)
treated = frame[frame.usable & (frame.wind > 0)].copy()
def control_mean(lat):
    sel = controls[(controls.lat >= lat - band) & (controls.lat <= lat + band)]
    return sel.delta.mean() if len(sel) >= 5 else None
treated['ctrl_delta'] = treated.lat.map(control_mean)
treated = treated.dropna(subset=['ctrl_delta'])
treated['did'] = treated.delta - treated.ctrl_delta
result = (treated.groupby('wind')
          .agg(fields=('did', 'size'), mean_delta=('delta', 'mean'),
               mean_ctrl=('ctrl_delta', 'mean'), did=('did', 'mean'),
               did_se=('did', 'sem'))
          .round(4))
result.index = result.index.map({1: '60-79 mph', 2: '80-99 mph', 3: '100+ mph'})
result

,fields,mean_delta,mean_ctrl,did,did_se
wind,,,,,
60-79 mph,527,-0.0116,0.0028,-0.0144,0.0028
80-99 mph,142,-0.0384,0.0028,-0.0412,0.0047
100+ mph,116,-0.0485,0.0075,-0.0561,0.0070


In [4]:
dids = result['did'].tolist()
assert all(b < a for a, b in zip(dids, dids[1:])), (
    'DiD is not monotonically worsening across wind classes: ' + str(dids))
print('Monotonicity holds: DiD worsens with wind class:', dids)

Monotonicity holds: DiD worsens with wind class: [-0.0144, -0.0412, -0.0561]


## Attrition: who the clouds dropped

Storms make clouds, so the fields the validity filter drops could correlate
with wind band; if they did, the surviving sample would be non-random. Counts
per band at each stage: all corn fields, fields passing the null/valid_frac
filter on both dates, and (treated bands only) fields matched to at least 5
controls. The match column is blank for controls by construction.

In [5]:
att = frame.groupby('wind').agg(corn_fields=('usable', 'size'),
                                cloud_ok=('usable', 'sum'))
att['matched'] = treated.groupby('wind').size()
att['dropped_pct'] = (100 * (1 - att.cloud_ok / att.corn_fields)).round(1)
att.index = att.index.map({0: 'control', 1: '60-79 mph', 2: '80-99 mph', 3: '100+ mph'})
att

,corn_fields,cloud_ok,matched,dropped_pct
wind,,,,
control,108,74,NaN,31.5
60-79 mph,758,527,527.0,30.5
80-99 mph,814,436,142.0,46.4
100+ mph,2058,954,116.0,53.6


## Reading the result

- The wind signal survives the drought control: NDVI loss deepens class by
  class after subtracting what matched-latitude unaffected fields did over the
  same 15 days.
- Magnitudes are conservative. Optical NDVI understates lodging because
  flattened corn stays green for weeks; SAR-based studies (Remote Sensing
  12(23):3878; BAMS 103(4)) found structural damage where NDVI shows only a
  modest dip. Treat these numbers as a floor, not the damage estimate.
- Controls are matched on latitude only. Soil, hybrid maturity, and local
  rainfall vary within a band; a production study would match on more.
- Attrition itself correlates with wind: 30-31% of control and 60-79 mph corn
  fields fail the validity filter vs 46% (80-99) and 54% (100+), residual
  storm cloud sitting over the harder-hit swath. If cloudier fields are also
  more damaged, survivors understate damage, consistent with reading these
  numbers as a floor; the sign of the bias is not provable from optical data
  alone.
- Single county (Benton), single sensor, two dates. The mvp and state scopes
  extend the same tables statewide; this notebook re-runs unchanged.